# **Imports**

In [1]:
from dataclasses import dataclass
from typing import Literal
import dataclasses
import glob
import os
import random
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, dataset

from torchvision import datasets
from torchvision.transforms import (
    Compose,
    Normalize,
    Resize,
    ToTensor,
    ToPILImage,
)

import wandb
from kaggle_secrets import UserSecretsClient

**Wandb Initilization**

In [2]:
user_secrets = UserSecretsClient()
wandb.login(key = user_secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ashiklibu1911 (ashiklibu1911-national-chung-cheng-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# **Configuration** 

In [3]:
from dataclasses import dataclass
import dataclasses
from typing import Literal


@dataclass
class TrainConfig:
    # Training
    epochs: int = 100
    lr: float = 1e-3
    device: Literal["cuda", "cpu"] = "cuda"

    # Checkpoints
    load_from_checkpoint: bool = False
    checkpoint_path: str = "./with_residual/checkpoints"

    save_best_model: bool = True
    save_last_model: bool = True

    early_stoping: bool = True
    patience_count: int = 5

    # Monitor metric for best model
    monitor: Literal["val_loss", "val_acc"] = "val_acc"

    # Plots
    save_plots: bool = True
    plot_path: str = "./with_residual/plots"

    # WandB
    wandb_monitor: bool = True
    project_name: str = "ResNet"
    run_name: str = "resnet-with-cifar-10"

    img_size: int = 224

    save_csv: bool = True
    csv_path: str = "./with_residual/log.csv"


@dataclass
class DatasetConfig:
    # Dataset
    img_size: int = 224
    batch_size: int = 32

    # DataLoader
    train_shuffle: bool = True
    test_shuffle: bool = False
    num_workers: int = 4
    pin_memory: bool = True

    # Transform
    normalize: bool = True


@dataclass
class InferenceConfig:
    img_size: int = 224
    load_from_checkpoint: bool = True
    checkpoint_path: str = "./with_residual/checkpoints/best.pt"
    dataset_images_path: str = "./with_residual/images"
    plot_file_name: str = "./with_residual/plots/sample.png"


@dataclass
class ModelConfig:
    model: Literal["resnet50", "resnet101", "resnet152"] = "resnet50"
    num_channels: int = 3
    num_classes: int = 10
    residual: bool = True

# **Model**

In [4]:
class ResidualBlock(nn.Module):
    def __init__(self, in_planes, planes, downsample = None, middel_conve_stride = 1, residual = True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = in_planes, out_channels = planes, kernel_size = 1, stride = middel_conve_stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(in_channels = planes, out_channels = planes, kernel_size = 3, stride = 1, padding = 1)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(in_channels = planes, out_channels = planes * 4, kernel_size = 1, stride = 1)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU()
        self.downsample = downsample
        self.residual = residual

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)
        

        if self.residual:
            if self.downsample is not None:
                x = self.downsample(x)
            out = out + x
        out = self.relu(out)
        return out



class ResNet(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        if config.model == "resnet50":
            self.layers_count = [3, 4, 6, 3]
        elif config.model == "resnet101":
            self.layers_count = [3, 4, 23, 3]
        else:
            self.layers_count = [3, 8, 36, 3]
        
        self.in_planes = 64
        self.model = nn.Sequential(
        nn.Conv2d(in_channels = 3, out_channels = self.in_planes, kernel_size = 7, stride = 2, padding = 3),   
        nn.BatchNorm2d(self.in_planes),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride = 2, padding = 1),

        self._make_layers(self.layers_count[0], planes = 64, stride = 1),
        self._make_layers(self.layers_count[1], planes = 128, stride = 2),
        self._make_layers(self.layers_count[2], planes = 256, stride = 2),
        self._make_layers(self.layers_count[3], planes = 512, stride = 2),
        nn.AdaptiveAvgPool2d((1, 1)))
        self.predictor = nn.Linear(2048, self.config.num_classes)


    def forward(self, x):
        x = self.model(x).flatten(1)
        return self.predictor(x)

    def _make_layers(self, num_residual_block, planes, stride):
        downsample = None
        layers = nn.ModuleList()

        if stride != 1 or self.in_planes != planes * 4:
            downsample = nn.Sequential(nn.Conv2d(self.in_planes, planes * 4, 1, stride = stride),
                            nn.BatchNorm2d(planes*4))
        layers.append(ResidualBlock(in_planes = self.in_planes,
                        planes = planes, 
                        downsample = downsample,
                        middel_conve_stride = stride,
                        residual = self.config.residual))
        self.in_planes = planes * 4
        for _ in range(num_residual_block -1):
            layers.append(ResidualBlock(in_planes = self.in_planes, planes = planes, residual = self.config.residual))
        return nn.Sequential(*layers)






# **Data class**

In [5]:
class Dataset:

    def __init__(self, config):
        self.config = config

        self._build_dataset()
        self._build_dataloader()

    def _build_transform(self):
        transforms = [
            Resize((self.config.img_size, self.config.img_size)),
            ToTensor(),
        ]

        # CIFAR-10 mean & std
        if self.config.normalize:
            transforms.append(
                Normalize(
                    mean=(0.4914, 0.4822, 0.4465),
                    std=(0.2470, 0.2435, 0.2616),
                )
            )

        return Compose(transforms)

    def _build_dataset(self):
        transform = self._build_transform()

        self.train_dataset = datasets.CIFAR10(
            root="./data",
            train=True,
            download=True,
            transform=transform,
        )

        self.test_dataset = datasets.CIFAR10(
            root="./data",
            train=False,
            download=True,
            transform=transform,
        )
        self.test_infer = datasets.CIFAR10(
            root="./data",
            train=False,
            download=True,
        )
        self.classes = self.train_dataset.classes
        self.idToclasses = {v: k for k, v in self.train_dataset.class_to_idx.items()}

    def _build_dataloader(self):

        self.train_loader = DataLoader(
            self.train_dataset,
            batch_size=self.config.batch_size,
            shuffle=self.config.train_shuffle,
            num_workers=self.config.num_workers,
            pin_memory=self.config.pin_memory,
        )

        self.test_loader = DataLoader(
            self.test_dataset,
            batch_size=self.config.batch_size,
            shuffle=self.config.test_shuffle,
            num_workers=self.config.num_workers,
            pin_memory=self.config.pin_memory,
        )

    def __len__(self):
        return len(self.train_dataset)

    def num_classes(self):
        return len(self.classes)

# **Trainer Class**

In [6]:
class Trainer:
    def __init__(self, config):
        self.config = config
        self.patience = 0
        self.device = torch.device(
            "cuda"
            if config.device == "cuda" and torch.cuda.is_available()
            else "cpu"
        )
        self.history = {
            "train_loss": [],
            "train_acc": [],
            "val_loss": [],
            "val_acc": [],
        }

        if self.config.wandb_monitor:
            wandb.init(project = self.config.project_name,
                        name = self.config.run_name,
                        config = vars(config))

        if self.config.monitor == "val_acc":
            self.best_metric = -float("inf")
        else:
            self.best_metric = float("inf")

    def train(self, model, dataset):

        model = model.to(self.device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(
            model.parameters(),
            lr=self.config.lr,
        )
        start_epoch = 0
        if self.config.load_from_checkpoint:
            start_epoch = self._load_checkpoint(
                model,
                optimizer,
            )

        for epoch in range(start_epoch, self.config.epochs):

            if self.patience == self.config.patience_count:
                continue

            print(f"\nEpoch [{epoch+1}/{self.config.epochs}]")

            train_loss, train_acc = self._train_epoch(
                model,
                dataset.train_loader,
                criterion,
                optimizer,
            )

            val_loss, val_acc, _, _ = self._validate_epoch(
                model,
                dataset.test_loader,
                criterion,
            )

            self.history["train_loss"].append(train_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_loss"].append(val_loss)
            self.history["val_acc"].append(val_acc)

            if self.config.wandb_monitor:
                wandb.log({"epoch" : epoch + 1,
                        "train/loss" : train_loss,
                        "train/accuracy" : train_acc,
                        "test/loss" : val_loss,
                        "test/accuracy": val_acc})
            print(
                f"Train Loss: {train_loss:.4f} | "
                f"Train Acc: {train_acc:.2f}% | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val Acc: {val_acc:.2f}%"
            )

            if self.config.save_last_model:
                self._save_checkpoint(
                    model,
                    optimizer,
                    epoch + 1,
                    "last.pt",
                )

            if self.config.save_best_model:

                metric = (
                    val_acc
                    if self.config.monitor == "val_acc"
                    else val_loss
                )

                improved = (
                    metric > self.best_metric
                    if self.config.monitor == "val_acc"
                    else metric < self.best_metric
                )

                if improved:
                    self.patience = 0
                    self.best_metric = metric

                    self._save_checkpoint(
                        model,
                        optimizer,
                        epoch + 1,
                        "best.pt",
                    )

                    if self.config.wandb_monitor:
                        wandb.log({
                            "best_val_accuracy": val_acc,
                            "best_val_loss" : val_loss
                        })
                elif self.config.early_stoping:
                    self.patience += 1
                    if self.patience == self.config.patience_count:
                        wandb.run.summary["early_stopping"] = True
                        wandb.run.summary["early_stopping_epoch"] = epoch
                        print(f"Early stoping trigered at epoch {epoch} due to no improvement")
                        break
                    

        _ = self._load_checkpoint(model, optimizer, best = True)

        if self.config.save_plots:
            self._plot_history()
            self._plot_confusion_matrix(model, dataset, criterion)
        
        if self.config.save_csv:
            self._save_csv()

        if self.config.wandb_monitor:
            self._upload_models_to_wandb()
            
        
        


        return self.history

    def _train_epoch(
        self,
        model,
        loader,
        criterion,
        optimizer,
    ):

        model.train()

        running_loss = 0
        correct = 0
        total = 0
        loop = tqdm(loader)
        for images, labels in loop:
            images = images.to(self.device)
            labels = labels.to(self.device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            predicted = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            loop.set_postfix(
                loss=running_loss / (loop.n + 1),
                acc=100 * correct / total,
            )
        epoch_loss = running_loss / len(loader)
        epoch_acc = 100 * correct / total
        return epoch_loss, epoch_acc

    @torch.no_grad()
    def _validate_epoch(
        self,
        model,
        loader,
        criterion,
    ):
        model.eval()
        running_loss = 0
        correct = 0
        total = 0
        actual, predict =[], []
        for images, labels in loader:
            images = images.to(self.device)
            labels = labels.to(self.device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            predicted = outputs.argmax(dim=1)
            actual.append(predicted)
            predict.append(labels)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        epoch_loss = running_loss / len(loader)
        epoch_acc = 100 * correct / total
        return epoch_loss, epoch_acc, actual, predict
    def _save_checkpoint(
        self,
        model,
        optimizer,
        epoch,
        filename,
    ):
        os.makedirs(
            self.config.checkpoint_path,
            exist_ok=True,
        )
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            },
            os.path.join(
                self.config.checkpoint_path,
                filename,
            ),
        )

    def _load_checkpoint(
        self,
        model,
        optimizer,
        best = False
    ):

        checkpoint = torch.load(
            os.path.join(
                self.config.checkpoint_path,
                "best.pt" if best else "last.pt",
            ),
            map_location=self.device,
        )

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )
        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )
        print("Checkpoint Loaded")
        return checkpoint["epoch"]

    def _upload_models_to_wandb(self):

        best_path = os.path.join(
            self.config.checkpoint_path,
            "best.pt"
        )

        last_path = os.path.join(
            self.config.checkpoint_path,
            "last.pt"
        )


        # Upload best model
        if os.path.exists(best_path):

            best_artifact = wandb.Artifact(
                name="best-model",
                type="model",
                description="Best validation performance model"
            )

            best_artifact.add_file(best_path)

            wandb.log_artifact(best_artifact)


        # Upload last model
        if os.path.exists(last_path):

            last_artifact = wandb.Artifact(
                name="last-model",
                type="model",
                description="Final epoch model"
            )

            last_artifact.add_file(last_path)

            wandb.log_artifact(last_artifact)
    
    def _plot_history(self):
        os.makedirs(
            self.config.plot_path,
            exist_ok=True,
        )
        epochs = range(
            1,
            len(self.history["train_loss"]) + 1,
        )
        plt.figure(figsize=(8, 5))
        plt.plot(
            epochs,
            self.history["train_loss"],
            label="Train",
        )
        plt.plot(
            epochs,
            self.history["val_loss"],
            label="Validation",
        )
        loss_path = os.path.join(self.config.plot_path, "loss.png")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Loss")
        plt.legend()
        plt.savefig(loss_path)

        plt.close()

        plt.figure(figsize=(8, 5))

        plt.plot(
            epochs,
            self.history["train_acc"],
            label="Train",
        )
        plt.plot(
            epochs,
            self.history["val_acc"],
            label="Validation",
        )
        acc_path = os.path.join(self.config.plot_path,"accuracy.png")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy (%)")
        plt.title("Accuracy")
        plt.legend()
        plt.savefig(acc_path)
        plt.close()

        if self.config.wandb_monitor:
            wandb.log({
                "loss_curve": wandb.Image(loss_path),
                "acc_curve" : wandb.Image(acc_path)
            })

    def _plot_confusion_matrix(self, model, dataset, criterion):
        idToClass = dataset.idToclasses
        _, _, actual, predict = self._validate_epoch(model, dataset.test_loader, criterion)

        actual = [i for batch in actual for i in batch]
        predict = [i for batch in predict for i in batch]
        cm = [[0 for _ in range(len(idToClass))] for _ in range(len(idToClass))]
        for i, j in zip(actual, predict):
            cm[i-1][j-1] += 1
        cm = torch.tensor(cm)
        class_names = [k for k in idToClass.keys()]

        plt.figure(figsize = (7, 6))

        sns.heatmap(
            cm,
            annot = True,
            fmt = "d",
            cmap = "Blues",
            xticklabels = class_names,
            yticklabels = class_names
        )


        confusion_mat_path = os.path.join(self.config.plot_path, "confusion_mat.png")
        plt.xlabel("predicted_labels")
        plt.ylabel("Actual labels")
        plt.title("Confusion Matrix")
        plt.tight_layout()
        plt.savefig(confusion_mat_path)
        plt.close()

        if self.config.wandb_monitor:
            wandb.log(
                {
                    "confussion_mat" : wandb.Image(confusion_mat_path)
                }
            )
    
    def _save_csv(self):
        df = pd.DataFrame([self.history])
        os.makedirs(os.path.dirname(self.config.csv_path), exist_ok = True)
        df.to_csv(self.config.csv_path)
        if self.config.wandb_monitor:
            csv_artifact = wandb.Artifact("Csv_log", type = "dataset", description = "log csv file")
            csv_artifact.add_file(self.config.csv_path)
            wandb.log_artifact(csv_artifact)

        


# **Inference**

In [7]:
class Classify:
    def __init__(self, model, config, loader):
        self.model = model
        self.config = config
        self.loader = loader
        self.transforms = Compose([ToTensor(),
                            Resize((self.config.img_size, self.config.img_size)),
                            Normalize(mean=(0.4914, 0.4822, 0.4465),\
                                 std=(0.2470, 0.2435, 0.2616))])
        self.resize = Resize((self.config.img_size, self.config.img_size))


    def predict(self, images: list | str | None = None):
        if images == None:
            images = self.pic_images_from_testset()
            images = glob.glob(self.config.dataset_images_path + "/*.jpg")
        if isinstance(images, list):
            fig, plots = plt.subplots(2, len(images)//2, figsize = (10, 5))
            plots = plots.flatten()
            if isinstance(images[0], str):
                for i in range(len(images)):
                    out = self._predict(images[i])
                    plots[i].imshow(Image.open(images[i]).convert("RGB"))
                    plots[i].set_title(f"Class : {self.loader.idToclasses[out]}")
                    plots[i].axis("off")
                plt.tight_layout()
                plt.savefig(self.config.plot_file_name)
                plt.close()
        elif isinstance(images, str):
            out = self._predict(images)
            plt.imshow(Image.open(images).convert("RGB"))
            plt.axis("off")
            plt.title(f"Class : {self.loader.idToclasses[out]}")
            plt.savefig(self.config.plot_file_name)
        else:
            print("Currently not supported")
        if wandb.run is not None and os.path.isfile(self.config.plot_file_name):
            wandb.log({
                "sample_prediction": wandb.Image(self.config.plot_file_name)
            })
            wandb.finish()

    
    def _predict(self, img):
        with torch.no_grad():
            img = Image.open(img).convert("RGB")
            img = self.transforms(img).unsqueeze(0)
            device = next(self.model.parameters()).device
            img = img.to(device)
            out = self.model(img).argmax()
            return int(out)

    def pic_images_from_testset(self, count = 10, clear = False):
        if clear and os.path.isdir(self.config.dataset_images_path):
            shutil.rmtree(self.config.dataset_images_path)
        test_set = self.loader.test_infer
        os.makedirs(self.config.dataset_images_path, exist_ok = True)
        total = len(test_set)
        for i in range(count):
            idx = random.randint(0, total-1)
            img , label = test_set[idx]
            img = self.resize(img)
            img.save(f"{self.config.dataset_images_path}/test_{i+1}.jpg")
        return
       

# **Pipeline**

In [8]:
data_config = DatasetConfig()
train_config = TrainConfig()
inference_config = InferenceConfig()
model_config = ModelConfig()
data = Dataset(data_config)
model = ResNet(model_config)
train = Trainer(train_config)
train.train(model, data)

state_dict = torch.load(os.path.join(train_config.checkpoint_path, "best.pt"))
predict = Classify(model, inference_config, data)
predict.predict()

100%|██████████| 170M/170M [31:21<00:00, 90.6kB/s]
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260814_080529-8np5g6ak
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run resnet-with-cifar-10
wandb: ⭐️ View project at https://wandb.ai/ashiklibu1911-national-chung-cheng-university/ResNet
wandb: 🚀 View run at https://wandb.ai/ashiklibu1911-national-chung-cheng-university/ResNet/runs/8np5g6ak



Epoch [1/100]


100%|██████████| 1563/1563 [08:16<00:00,  3.15it/s, acc=41.3, loss=1.6]


Train Loss: 1.6017 | Train Acc: 41.35% | Val Loss: 1.2430 | Val Acc: 55.89%

Epoch [2/100]


100%|██████████| 1563/1563 [08:23<00:00,  3.10it/s, acc=62.8, loss=1.04]


...

100%|██████████| 1563/1563 [08:21<00:00,  3.11it/s, acc=97.5, loss=0.073]


Train Loss: 0.0730 | Train Acc: 97.46% | Val Loss: 0.6738 | Val Acc: 83.29%

Epoch [19/100]


100%|██████████| 1563/1563 [08:20<00:00,  3.12it/s, acc=97.9, loss=0.0617]


Train Loss: 0.0617 | Train Acc: 97.88% | Val Loss: 0.7185 | Val Acc: 83.32%
Early stoping trigered at epoch 18 due to no improvement
Checkpoint Loaded


wandb: uploading artifact best-model; uploading artifact last-model; updating run metadata
wandb: uploading artifact best-model; uploading artifact last-model
wandb: uploading artifact last-model
wandb: 
wandb: Run history:
wandb: best_val_accuracy ▁▃▅▆▇▇███
wandb:     best_val_loss █▆▃▃▂▁▁▁▂
wandb:             epoch ▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇██
wandb:     test/accuracy ▁▃▅▆▇▇▇█████████▇██
wandb:         test/loss █▆▃▃▂▁▂▁▁▂▂▂▂▂▃▃▃▃▃
wandb:    train/accuracy ▁▄▅▅▆▆▇▇▇▇▇████████
wandb:        train/loss █▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:    best_val_accuracy 84.5
wandb:        best_val_loss 0.5967
wandb:       early_stopping True
wandb: early_stopping_epoch 18
wandb:                epoch 19
wandb:        test/accuracy 83.32
wandb:            test/loss 0.71847
wandb:       train/accuracy 97.88
wandb:           train/loss 0.06174
wandb: 
wandb: 🚀 View run resnet-with-cifar-10 at: https://wandb.ai/ashiklibu1911-national-chung-cheng-university/ResNet/runs/8np5g6ak
wandb: ⭐️ View 

# **Without Residual**

In [9]:
from dataclasses import dataclass
import dataclasses
from typing import Literal


@dataclass
class TrainConfig:
    # Training
    epochs: int = 100
    lr: float = 1e-3
    device: Literal["cuda", "cpu"] = "cuda"

    # Checkpoints
    load_from_checkpoint: bool = False
    checkpoint_path: str = "./without_residual/checkpoints"

    save_best_model: bool = True
    save_last_model: bool = True

    early_stoping: bool = True
    patience_count: int = 5

    # Monitor metric for best model
    monitor: Literal["val_loss", "val_acc"] = "val_acc"

    # Plots
    save_plots: bool = True
    plot_path: str = "./without_residual/plots"

    # WandB
    wandb_monitor: bool = True
    project_name: str = "ResNet"
    run_name: str = "resnet-without-cifar-10"

    img_size: int = 224

    save_csv: bool = True
    csv_path: str = "./without_residual/log.csv"


@dataclass
class DatasetConfig:
    # Dataset
    img_size: int = 224
    batch_size: int = 32

    # DataLoader
    train_shuffle: bool = True
    test_shuffle: bool = False
    num_workers: int = 4
    pin_memory: bool = True

    # Transform
    normalize: bool = True


@dataclass
class InferenceConfig:
    img_size: int = 224
    load_from_checkpoint: bool = True
    checkpoint_path: str = "./without_residual/checkpoints/best.pt"
    dataset_images_path: str = "./without_residual/images"
    plot_file_name: str = "./without_residual/plots/sample.png"


@dataclass
class ModelConfig:
    model: Literal["resnet50", "resnet101", "resnet152"] = "resnet50"
    num_channels: int = 3
    num_classes: int = 10
    residual: bool = False

In [10]:
data_config = DatasetConfig()
train_config = TrainConfig()
inference_config = InferenceConfig()
model_config = ModelConfig()
data = Dataset(data_config)
model = ResNet(model_config)
train = Trainer(train_config)
train.train(model, data)

state_dict = torch.load(os.path.join(train_config.checkpoint_path, "best.pt"))
predict = Classify(model, inference_config, data)
predict.predict()

wandb: setting up run dpdhs8vi
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260814_105551-dpdhs8vi
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run resnet-without-cifar-10
wandb: ⭐️ View project at https://wandb.ai/ashiklibu1911-national-chung-cheng-university/ResNet
wandb: 🚀 View run at https://wandb.ai/ashiklibu1911-national-chung-cheng-university/ResNet/runs/dpdhs8vi



Epoch [1/100]


100%|██████████| 1563/1563 [06:51<00:00,  3.80it/s, acc=20.4, loss=2.05]


Train Loss: 2.0508 | Train Acc: 20.41% | Val Loss: 2.4069 | Val Acc: 22.87%

Epoch [2/100]


100%|██████████| 1563/1563 [06:49<00:00,  3.81it/s, acc=27.3, loss=1.9]


...

100%|██████████| 1563/1563 [06:54<00:00,  3.77it/s, acc=88.1, loss=0.34]


Train Loss: 0.3403 | Train Acc: 88.07% | Val Loss: 1.0189 | Val Acc: 70.72%

Epoch [28/100]


100%|██████████| 1563/1563 [06:54<00:00,  3.77it/s, acc=89.4, loss=0.302]


Train Loss: 0.3017 | Train Acc: 89.42% | Val Loss: 0.9593 | Val Acc: 72.16%
Early stoping trigered at epoch 27 due to no improvement
Checkpoint Loaded


wandb: uploading artifact best-model; uploading artifact last-model; updating run metadata
wandb: uploading artifact best-model; uploading artifact last-model
wandb: uploading artifact last-model
wandb: 
wandb: Run history:
wandb: best_val_accuracy ▁▂▃▃▄▅▅▆▆▇▇▇▇██████
wandb:     best_val_loss █▆▅▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁
wandb:             epoch ▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
wandb:     test/accuracy ▁▂▂▃▃▃▄▅▅▆▆▆▇▇▇▇████████████
wandb:         test/loss █▆▆▅▄▅▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▂
wandb:    train/accuracy ▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
wandb:        train/loss █▇▇▇▇▆▆▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁
wandb: 
wandb: Run summary:
wandb:    best_val_accuracy 73.2
wandb:        best_val_loss 0.82151
wandb:       early_stopping True
wandb: early_stopping_epoch 27
wandb:                epoch 28
wandb:        test/accuracy 72.16
wandb:            test/loss 0.95933
wandb:       train/accuracy 89.418
wandb:           train/loss 0.30173
wandb: 
wandb: 🚀 View run resnet-without-cifar-10 at: https://wandb.ai/ashiklibu191